# Trompo 3D con ángulos de Euler — Plotly + NumPy

**Para Google Colab, sin VPython ni `FigureWidget`.**

Este cuaderno visualiza un trompo con la convención **ZXZ activa**: 

$$R(\phi,\theta,\psi)=R_z(\phi)R_x(\theta)R_z(\psi).$$

Muestra en cada instante:

$$R,\qquad\dot R,\qquad\underbrace{R^T\dot R}_{[\omega_{cuerpo}]_\times},\qquad\underbrace{\dot R R^T}_{[\omega_{lab}]_\times}.$$

**Cómo usarlo:** ejecuta las celdas en orden; ve a la última celda de visualización, mueve las seis barras de Colab y **vuelve a ejecutar esa celda**. En **Animación**, usa ▶ Reproducir y ❚❚ Pausar para ver cambiar simultáneamente el trompo, los vectores y las cuatro matrices. Puedes girar el punto de vista arrastrando la figura con el mouse.

> **Importante:** a diferencia de VPython, las barras `#@param` de Colab aplican los cambios cuando se ejecuta la celda, no de forma continua mientras se arrastran. Una vez generada, la animación de Plotly se reproduce en el navegador sin enviar cada fotograma a Python.


## 0. Instalación

Colab normalmente ya tiene NumPy y Plotly. La siguiente celda actualiza Plotly si es necesario.

In [ ]:
# En Colab puedes ejecutar esta celda una sola vez.
%pip -q install "plotly>=5.20" numpy


## 1. Bibliotecas y convención de rotación

Los ángulos y velocidades que elegimos están en **grados** y **grados/s**; para derivar matrices usamos **radianes** y **radianes/s**. La matriz $R$ transforma coordenadas del cuerpo en coordenadas del laboratorio.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

# En Google Colab seleccionamos su renderizador nativo.
# No utilizamos FigureWidget ni extensiones personalizadas.
import sys
import plotly.io as pio
if "google.colab" in sys.modules:
    pio.renderers.default = "colab"

print("NumPy:", np.__version__)
print("Plotly:", go.__version__ if hasattr(go, "__version__") else __import__("plotly").__version__)


## 2. Cinemática: matrices y velocidad angular

Usamos los generadores:

$$J_z=\begin{pmatrix}0&-1&0\\1&0&0\\0&0&0\end{pmatrix},\qquad J_x=\begin{pmatrix}0&0&0\\0&0&-1\\0&1&0\end{pmatrix}.$$

La derivada se obtiene de la regla del producto:

$$\dot R=\dot\phi J_zR + \dot\theta R_z(\phi)J_xR_x(\theta)R_z(\psi)+\dot\psi RJ_z.$$

Las matrices $R^T\dot R$ y $\dot RR^T$ son antisimétricas. Representan **el mismo vector físico de velocidad angular** en bases diferentes. Para $R$ que transforma cuerpo a laboratorio:

$$[\omega_{cuerpo}]_\times=R^T\dot R,\quad [\omega_{lab}]_\times=\dot RR^T,\quad\omega_{lab}=R\omega_{cuerpo}.$$

In [ ]:
# ============================================================
# 1. ROTACIONES DE EULER: CONVENCION ZXZ
# ============================================================
# Convencion activa: r_laboratorio = R @ r_cuerpo.
# Los angulos se ingresan en grados; NumPy trabaja en radianes.

def Rz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0.0],
                     [s,  c, 0.0],
                     [0.0, 0.0, 1.0]])


def Rx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1.0, 0.0, 0.0],
                     [0.0, c, -s],
                     [0.0, s,  c]])


# Generadores infinitesimales del grupo de rotaciones SO(3).
Jz = np.array([[0.0, -1.0, 0.0],
               [1.0,  0.0, 0.0],
               [0.0,  0.0, 0.0]])

Jx = np.array([[0.0, 0.0,  0.0],
               [0.0, 0.0, -1.0],
               [0.0, 1.0,  0.0]])


def vee(W):
    """Extrae omega de la matriz antisimetrica [omega]_x."""
    return np.array([W[2, 1], W[0, 2], W[1, 0]])


def calcular_euler(phi, theta, psi, phi_dot=0, theta_dot=0, psi_dot=0):
    """
    Entrada: angulos en grados y velocidades en grados/segundo.
    Salida: R, dR/dt, matrices de velocidad angular y vectores.
    Las velocidades angulares de salida estan en rad/s.
    """
    p, th, q = np.deg2rad([phi, theta, psi])
    dp, dth, dq = np.deg2rad([phi_dot, theta_dot, psi_dot])

    A, B, C = Rz(p), Rx(th), Rz(q)
    R = A @ B @ C

    # Derivada ANALITICA: regla del producto, sin diferencias finitas.
    dR = (dp * Jz @ R
          + dth * A @ Jx @ B @ C
          + dq * R @ Jz)

    Omega_cuerpo = R.T @ dR   # R^T dR/dt
    Omega_lab = dR @ R.T      # dR/dt R^T
    w_cuerpo = vee(Omega_cuerpo)
    w_lab = vee(Omega_lab)

    # Ejes de la base movil del trompo.
    e1, e2, e3 = R[:, 0], R[:, 1], R[:, 2]
    k = np.array([0.0, 0.0, 1.0])
    linea_nodos = A @ np.array([1.0, 0.0, 0.0])

    # Descomposicion geometrica de la velocidad angular en laboratorio:
    # omega = phi_dot * k + theta_dot * n + psi_dot * e3.
    w_phi = dp * k
    w_theta = dth * linea_nodos
    w_psi = dq * e3

    return dict(Rz_phi=A, Rx_theta=B, Rz_psi=C,
                R=R, dR=dR, Omega_cuerpo=Omega_cuerpo,
                Omega_lab=Omega_lab, w_cuerpo=w_cuerpo,
                w_lab=w_lab, e1=e1, e2=e2, e3=e3,
                nodos=linea_nodos, w_phi=w_phi,
                w_theta=w_theta, w_psi=w_psi)


## 3. Geometría del trompo

Se discretiza el perfil del sólido mediante anillos. Las caras triangulares se dibujan con `go.Mesh3d`.

La **marca naranja** permite distinguir $\psi$ incluso cuando el trompo es simétrico. Los ejes del laboratorio son X rojo, Y verde y Z azul. El eje móvil $e_3$ es blanco. La **flecha dorada** representa $\omega$ en el laboratorio.

In [ ]:
# ============================================================
# 2. GEOMETRIA DEL TROMPO (SE CREA UNA SOLA VEZ)
# ============================================================
# Perfil (altura z, radio r) del trompo en su propia base.
# La punta esta cerca del origen; el eje de simetria es el eje z.
PERFIL = np.array([
    [0.00, 0.04],
    [0.18, 0.13],
    [0.37, 0.26],
    [0.63, 0.44],
    [0.92, 0.65],
    [1.24, 0.80],
    [1.43, 0.81],
    [1.52, 0.66],
    [1.62, 0.23],
    [1.73, 0.085],
    [2.43, 0.085],
    [2.48, 0.035],
])

N_CIRC = 30
angulos = np.linspace(0, 2*np.pi, N_CIRC, endpoint=False)

puntos = []
for z, radio in PERFIL:
    for a in angulos:
        puntos.append([radio*np.cos(a), radio*np.sin(a), z])
P = np.asarray(puntos).T  # 3 x numero_de_vertices

# Conectamos anillos vecinos formando triangulos del mallado.
i_faces, j_faces, k_faces = [], [], []
for h in range(len(PERFIL) - 1):
    for j in range(N_CIRC):
        jp = (j + 1) % N_CIRC
        a, b = h*N_CIRC+j, h*N_CIRC+jp
        c, d = (h+1)*N_CIRC+j, (h+1)*N_CIRC+jp
        # Sentido exterior de las caras para iluminar correctamente.
        i_faces.extend([a, b])
        j_faces.extend([b, d])
        k_faces.extend([c, c])

# Marcador radial para que el giro propio psi sea visible.
marcador_local = np.array([[0.0, 0.79],
                           [0.0, 0.0],
                           [1.51, 1.51]])

COLORES = dict(trompo="#2086BF", naranja="#FF9F32",
                eje1="#EBCB55", eje2="#BB91FF", eje3="#EEEEEE",
                nodo="#F782D1", omega="#E7AE19",
                precesion="#FF62B0", nutacion="#51E2C5",
                propio="#FF9541")


def linea(origen, extremo, color, nombre, grosor=6, visible=True):
    """Trazo 3D con un punto visible en el extremo del vector."""
    o, e = np.asarray(origen), np.asarray(extremo)
    distancia = np.linalg.norm(e - o)
    return go.Scatter3d(
        x=[o[0], e[0]], y=[o[1], e[1]], z=[o[2], e[2]],
        mode="lines+markers", name=nombre,
        line=dict(color=color, width=grosor),
        marker=dict(color=color, size=[1.0, 5.5], symbol="circle"),
        visible=(visible and distancia > 1e-7),
        hovertemplate=nombre+"<extra></extra>",
        showlegend=False)


def trazas_moviles(datos):
    """Estas diez trazas son las unicas que cambian en cada instante."""
    R = datos["R"]
    v = R @ P
    marca = R @ marcador_local
    e1, e2, e3 = datos["e1"], datos["e2"], datos["e3"]

    cuerpo = go.Mesh3d(
        x=v[0], y=v[1], z=v[2],
        i=i_faces, j=j_faces, k=k_faces,
        color=COLORES["trompo"], opacity=0.96,
        flatshading=False,
        lighting=dict(ambient=0.65, diffuse=0.75,
                      specular=0.48, roughness=0.42),
        name="Trompo", hoverinfo="skip", showlegend=False)
    marca_traza = linea(marca[:, 0], marca[:, 1],
                        COLORES["naranja"], "Marca del giro propio", 12)

    # Base movil: trasladamos e1 y e2 un poco hacia el centro del trompo.
    b = 0.43*e3
    tr_e1 = linea(b, b+1.05*e1, COLORES["eje1"], "Eje movil e1", 5)
    tr_e2 = linea(b, b+1.05*e2, COLORES["eje2"], "Eje movil e2", 5)
    tr_e3 = linea([0, 0, 0], 2.75*e3, COLORES["eje3"], "Eje movil e3", 6)
    tr_n = linea([0, 0, 0], 1.25*datos["nodos"],
                 COLORES["nodo"], "Linea de nodos", 4)

    # Flechas de las velocidades: longitud grafica ~ magnitud en rad/s.
    # Se dibujan DESDE un punto comun, pero no se suman punta con cola.
    b_omega = 0.30*e3
    escala = 0.57
    tr_w = linea(b_omega, b_omega+escala*datos["w_lab"],
                 COLORES["omega"], "omega total", 10)
    tr_wp = linea(b_omega, b_omega+escala*datos["w_phi"],
                  COLORES["precesion"], "omega precesion", 5)
    tr_wt = linea(b_omega, b_omega+escala*datos["w_theta"],
                  COLORES["nutacion"], "omega nutacion", 5)
    tr_ws = linea(b_omega, b_omega+escala*datos["w_psi"],
                  COLORES["propio"], "omega giro propio", 5)

    return [cuerpo, marca_traza, tr_e1, tr_e2, tr_e3,
            tr_n, tr_w, tr_wp, tr_wt, tr_ws]


def trazas_fijas():
    """Sistema de referencia del laboratorio y circulo horizontal."""
    x = linea([0, 0, 0], [2.4, 0, 0], "#EF5555", "X fijo", 6)
    y = linea([0, 0, 0], [0, 2.4, 0], "#46C16C", "Y fijo", 6)
    z = linea([0, 0, 0], [0, 0, 2.4], "#69C1FA", "Z fijo", 6)
    ang = np.linspace(0, 2*np.pi, 80)
    base = go.Scatter3d(x=1.4*np.cos(ang), y=1.4*np.sin(ang),
                        z=-0.025+0*ang, mode="lines",
                        line=dict(color="#60718C", width=2, dash="dot"),
                        hoverinfo="skip", showlegend=False)
    return [x, y, z, base]


## 4. Gráfica Plotly y matrices en cada fotograma

La animación está **precalculada en Python**. Plotly mantiene los controles ▶ Reproducir, ❚❚ Pausar y un deslizador temporal, y actualiza los objetos 3D y las matrices mostradas a la derecha.

Las flechas rosa, turquesa y naranja son las contribuciones de precesión, nutación y giro propio:

$$\boldsymbol\omega=\dot\phi\,\hat{\mathbf Z}+\dot\theta\,\mathbf n+\dot\psi\,\mathbf e_3.$$

In [ ]:
# ============================================================
# 3. PANEL DE MATRICES Y FIGURA 3D
# ============================================================

def matriz_html(M, nd=2):
    """Texto monoespaciado para una matriz 3x3 dentro de Plotly."""
    filas = []
    for fila in M:
        texto = "  ".join(f"{valor:+6.{nd}f}" for valor in fila)
        filas.append(texto.replace(" ", "&nbsp;"))
    return "<br>".join(filas)


def anotaciones(datos, t, angulos, velocidades):
    """Las matrices se actualizan al cambiar de fotograma."""
    detalles = [
        (0.98, "R (cuerpo → laboratorio)", datos["R"]),
        (0.77, "dR/dt (1/s)", datos["dR"]),
        (0.56, "Rᵀ dR/dt: Ω del cuerpo (1/s)", datos["Omega_cuerpo"]),
        (0.35, "dR/dt Rᵀ: Ω laboratorio (1/s)", datos["Omega_lab"]),
    ]
    anns = []
    for posicion_y, titulo, M in detalles:
        anns.append(dict(
            x=0.707, y=posicion_y, xref="paper", yref="paper",
            xanchor="left", yanchor="top", showarrow=False,
            align="left", font=dict(family="Courier New, monospace",
                                    size=15, color="#ECF2FA"),
            text=f"<b>{titulo}</b><br>" + matriz_html(M)))

    wl, wc = datos["w_lab"], datos["w_cuerpo"]
    a, b, c = angulos
    ad, bd, cd = velocidades
    contenido = (
        f"<b>t = {t:.2f} s</b> | "
        f"φ = {a%360:.1f}°, θ = {b:.1f}°, ψ = {c%360:.1f}°<br>"
        f"<b>ω laboratorio</b> = ({wl[0]:+.2f}, {wl[1]:+.2f}, {wl[2]:+.2f}) rad/s<br>"
        f"<b>ω cuerpo</b> = ({wc[0]:+.2f}, {wc[1]:+.2f}, {wc[2]:+.2f}) rad/s<br>"
        f"φ̇ = {ad:.0f}°/s, θ̇ = {bd:.0f}°/s, ψ̇ = {cd:.0f}°/s"
    )
    anns.append(dict(x=0.707, y=0.12, xref="paper", yref="paper",
                     xanchor="left", yanchor="top", showarrow=False,
                     align="left", font=dict(family="Arial", size=13,
                                             color="#D2E3F6"),
                     text=contenido))
    return anns


def crear_figura(phi0=40, theta0=35, psi0=0,
                 phi_dot=30, theta_dot=0, psi_dot=120,
                 duracion=7.0, n_fotogramas=80, animar=True):
    """
    Construye un trompo animado SOLO con Python + Plotly.
    Las matrices y las superficies se precalculan en NumPy.
    El navegador reproduce los fotogramas sin volver a ejecutar Python.
    """
    velocidades = (phi_dot, theta_dot, psi_dot)
    datos0 = calcular_euler(phi0, theta0, psi0, *velocidades)

    # Orden de las trazas: diez objetos moviles + cuatro elementos fijos.
    base_movil = trazas_moviles(datos0)
    objetos_fijos = trazas_fijas()
    fig = go.Figure(data=base_movil + objetos_fijos)
    dinamicas = list(range(len(base_movil)))

    fig.update_layout(
        title=dict(text="Ángulos de Euler ZXZ — trompo y velocidad angular",
                   x=0.02, font=dict(size=22, color="#E8EDF5")),
        template="plotly_dark", paper_bgcolor="#101825",
        margin=dict(l=8, r=8, t=75, b=88),
        width=1250, height=765,
        scene=dict(
            domain=dict(x=[0.0, 0.68], y=[0.0, 1.0]),
            bgcolor="#101825", aspectmode="cube",
            xaxis=dict(range=[-3.0, 3.0], title="X", gridcolor="#354055",
                       zerolinecolor="#8490A0"),
            yaxis=dict(range=[-3.0, 3.0], title="Y", gridcolor="#354055",
                       zerolinecolor="#8490A0"),
            zaxis=dict(range=[-2.5, 3.0], title="Z", gridcolor="#354055",
                       zerolinecolor="#8490A0"),
            camera=dict(eye=dict(x=1.7, y=-2.0, z=1.45),
                        up=dict(x=0, y=0, z=1)),
            uirevision="conservar-camara"),
        uirevision="conservar-camara",
        annotations=anotaciones(datos0, 0.,
                                (phi0, theta0, psi0), velocidades))

    if not animar:
        return fig

    tiempos = np.linspace(0, float(duracion), int(n_fotogramas))
    fotogramas = []
    pasos = []
    for indice, tiempo in enumerate(tiempos):
        ph = phi0 + phi_dot*tiempo
        th = theta0 + theta_dot*tiempo
        ps = psi0 + psi_dot*tiempo
        datos = calcular_euler(ph, th, ps, *velocidades)
        nombre = f"{indice:03d}"
        frame = go.Frame(
            name=nombre, data=trazas_moviles(datos), traces=dinamicas,
            layout=go.Layout(annotations=anotaciones(
                datos, tiempo, (ph, th, ps), velocidades)))
        fotogramas.append(frame)
        pasos.append(dict(
            label=f"{tiempo:.1f}", method="animate",
            args=[[nombre], dict(mode="immediate",
                                 frame=dict(duration=0, redraw=True),
                                 transition=dict(duration=0))]))

    fig.frames = tuple(fotogramas)
    fig.update_layout(
        updatemenus=[dict(
            type="buttons", direction="left", showactive=False,
            x=0.05, y=-0.07, xanchor="left", yanchor="top",
            bgcolor="#263B56", font=dict(color="white", size=15),
            buttons=[
                dict(label="▶ Reproducir", method="animate",
                     args=[None, dict(mode="immediate", fromcurrent=True,
                                     frame=dict(duration=85, redraw=True),
                                     transition=dict(duration=0))]),
                dict(label="❚❚ Pausar", method="animate",
                     args=[[None], dict(mode="immediate",
                                        frame=dict(duration=0, redraw=False),
                                        transition=dict(duration=0))])])],
        sliders=[dict(
            active=0, x=0.39, len=0.57, y=-0.05,
            xanchor="left", yanchor="top",
            pad=dict(t=12, b=6),
            currentvalue=dict(prefix="Fotograma: ", visible=True,
                              font=dict(color="#D8E5F6", size=12)),
            steps=pasos)])
    return fig


## 5. ¡Experimenta! Controles para Colab

En **Google Colab**, el bloque siguiente muestra los controles de ángulos y velocidades como **barras**. Cambia sus valores y pulsa ▶ en **la propia celda** para regenerar el escenario. Después pulsa **▶ Reproducir dentro de la figura** para observar el movimiento. En VS Code o Jupyter local cambia manualmente los seis números.

**Experimentos sugeridos:**

- $\dot\phi=0,\dot\theta=0,\dot\psi\neq0$: solo giro propio; la marca naranja rota.
- $\dot\phi\neq0,\dot\theta=0,\dot\psi=0$: precesión alrededor del eje Z.
- $\dot\theta\neq0$: nutación.
- $\theta=0$: singularidad de la parametrización ZXZ; los ángulos $\phi$ y $\psi$ dejan de ser independientes para describir la orientación.

> En el modo **Exploración** se dibuja una orientación estática con las matrices asociadas a las velocidades elegidas. En el modo **Animación** las matrices también varían con el tiempo.

In [ ]:
# ============================================================
# 4. CONTROLES DE COLAB: AJUSTA Y VUELVE A EJECUTAR ESTA CELDA
# ============================================================
# En Colab, los #@param se muestran como barras deslizantes.
# Para aplicar valores nuevos, pulsa ▶ en esta celda.
# En VS Code/Jupyter son variables Python normales (tambien funciona).

phi = 40      #@param {type:"slider", min:0, max:360, step:5}
theta = 35    #@param {type:"slider", min:0, max:180, step:5}
psi = 0       #@param {type:"slider", min:0, max:360, step:5}

phi_punto = 30      #@param {type:"slider", min:-180, max:180, step:10}
theta_punto = 0     #@param {type:"slider", min:-90, max:90, step:5}
psi_punto = 120     #@param {type:"slider", min:-360, max:360, step:10}

# Control de la duracion; Plotly reproduce los fotogramas en el navegador.
duracion = 7        #@param {type:"slider", min:3, max:12, step:1}
modo = "Animación"  #@param ["Animación", "Exploración"]

fig = crear_figura(
    phi0=phi, theta0=theta, psi0=psi,
    phi_dot=phi_punto, theta_dot=theta_punto, psi_dot=psi_punto,
    duracion=duracion, n_fotogramas=80,
    animar=(modo == "Animación"))

# El renderizado de Plotly no depende de widgets personalizados.
fig.show()


## 6. Verificación física y matemática

Comprobamos que $R^TR=I$, que $R^T\dot R$ es antisimétrica y que las velocidades de los dos sistemas de referencia se relacionan mediante $\omega_{lab}=R\omega_{cuerpo}$. Se compara además el resultado matricial con la fórmula analítica de Euler ZXZ:

$$\begin{aligned}
\omega_1&=\dot\phi\sin\theta\sin\psi+\dot\theta\cos\psi,\\
\omega_2&=\dot\phi\sin\theta\cos\psi-\dot\theta\sin\psi,\\
\omega_3&=\dot\phi\cos\theta+\dot\psi.
\end{aligned}$$

In [ ]:
# ============================================================
# 5. COMPROBACIONES MATEMATICAS (OPCIONAL)
# ============================================================
# (i) R es ortogonal; (ii) R^T dR es antisimetrica;
# (iii) omega laboratorio = R @ omega cuerpo.

d = calcular_euler(40, 35, 50, 30, 5, 120)
R = d["R"]
error_ortogonal = np.linalg.norm(R.T @ R - np.eye(3))
error_antisimetria = np.linalg.norm(d["Omega_cuerpo"] + d["Omega_cuerpo"].T)
error_omega = np.linalg.norm(d["w_lab"] - R @ d["w_cuerpo"])

print("|| R^T R - I || =", f"{error_ortogonal:.2e}")
print("|| Omega + Omega^T || =", f"{error_antisimetria:.2e}")
print("|| omega_lab - R omega_cuerpo || =", f"{error_omega:.2e}")

# Fórmula analítica en la base del cuerpo (Euler ZXZ).
p, th, q = np.deg2rad([40, 35, 50])
dp, dth, dq = np.deg2rad([30, 5, 120])
w_teorica = np.array([
    dp*np.sin(th)*np.sin(q) + dth*np.cos(q),
    dp*np.sin(th)*np.cos(q) - dth*np.sin(q),
    dp*np.cos(th) + dq
])
print("omega cuerpo (matrices) =", np.round(d["w_cuerpo"], 5), "rad/s")
print("omega cuerpo (fórmula)  =", np.round(w_teorica, 5), "rad/s")
assert np.allclose(w_teorica, d["w_cuerpo"])
assert error_ortogonal < 1e-10
assert error_antisimetria < 1e-10
assert error_omega < 1e-10


In [ ]:
# ============================================================
# 6. OPCIONAL: EXPORTAR LA ANIMACION COMO HTML INTERACTIVO
# ============================================================
# Incluye Plotly.js dentro del archivo: abre sin internet.
# El archivo conserva Play / Pausa, camara 3D y matrices dinamicas.
# Si el archivo resulta grande, usa include_plotlyjs="cdn" (requiere red).

# fig.write_html("Trompo_Euler_Plotly_Interactivo.html", include_plotlyjs=True)
# print("Archivo exportado correctamente")
